# Real-Time Live Geosteering Pipeline with Causal Extended Kalman Filter

## 1. Architectural & Scientific Principles
This pipeline implements a fully **causal, online real-time geosteering tracker** designed to guide well placement foot-by-foot during active drilling. It operates without looking forward in time, relying on two core engines:

### A. Curved Dipping Plane Engine (Structural Baseline)
Fits a quadratic dipping plane locally per-well using only spatial coordinates $(X, Y, Z)$ from the vertical build section, establishing a stable spatial trend baseline:
$$\text{TVT}_{\text{trend}} = f(X, Y, Z)$$

### B. Extended Kalman Filter Tracker (Causal Stratigraphic Tracking)
We model the stratigraphic deviation relative to the trend plane as a dynamic state $x_k = TVT_k - TVT_{\text{trend}, k}$:
1. **Transition Model:**
   $$x_k = x_{k-1} + w_k, \quad w_k \sim N(0, Q)$$
2. **Measurement Model (Typewell Correlation):**
   $$z_k = GR_{\text{obs}, k} = GR_{\text{typewell}}(TVT_{\text{trend}, k} + x_k) + v_k, \quad v_k \sim N(0, R)$$
3. **Kalman Gain & Update:**
   We linearize the reference log locally to obtain the stratigraphic Gamma Ray gradient:
   $$H_k = \left. \frac{d GR_{\text{typewell}}}{d TVT} \right|_{TVT = \hat{TVT}_{k\vert k-1}}$$
   $$K_k = \frac{P_{k\vert k-1} H_k}{H_k^2 P_{k\vert k-1} + R}$$
   $$\hat{x}_{k\vert k} = \hat{x}_{k\vert k-1} + K_k (GR_{\text{obs}, k} - GR_{\text{typewell}}(\hat{TVT}_{k\vert k-1}))$$

In [ ]:
import os, glob, time
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures
from scipy.interpolate import interp1d
from scipy.signal import savgol_filter

print('[*] Loading directories...')
def find_data_dirs():
    kaggle_input = '/kaggle/input'
    if os.path.exists(kaggle_input):
        for root, dirs, files in os.walk(kaggle_input):
            if 'train' in dirs and 'test' in dirs:
                return os.path.join(root, 'train'), os.path.join(root, 'test')
    for parent in ['.', '..', '../..']:
        t = os.path.join(parent, 'train')
        te = os.path.join(parent, 'test')
        if os.path.exists(t) and os.path.exists(te):
            if glob.glob(os.path.join(t, '*_horizontal_well.csv')):
                return t, te
    raise ValueError('Data directories not found')

train_dir, test_dir = find_data_dirs()
print(f'[+] Train: {train_dir}, Test: {test_dir}')

In [ ]:
def resolve_col(df, opts):
    for o in opts:
        if o in df.columns: return o
    for o in opts:
        for c in df.columns:
            if c.lower() == o.lower(): return c
    return None

def parse_wellname(fp):
    bn = os.path.basename(fp)
    return bn.split('__horizontal_well.csv')[0] if '__horizontal_well.csv' in bn else bn.split('_horizontal_well.csv')[0]

test_files = sorted(glob.glob(os.path.join(test_dir, '*_horizontal_well.csv')))
print(f'[+] Found {len(test_files)} test wells.')

In [ ]:
# ============================================================
# REAL-TIME CAUSAL CACHING & TRACKING ENGINE
# ============================================================
submission_rows = []

for f in test_files:
    wellname = parse_wellname(f)
    df = pd.read_csv(f)
    
    tw_file = None
    for sep in ['__', '_']:
        tf = os.path.join(test_dir, f'{wellname}{sep}typewell.csv')
        if os.path.exists(tf):
            tw_file = tf
            break
            
    if not tw_file:
        print(f'[-] Warning: Typewell not found for {wellname}, skipping.')
        continue
        
    t_df = pd.read_csv(tw_file)
    
    h_md = resolve_col(df, ['MD', 'Depth', 'DEPTH'])
    h_x = resolve_col(df, ['X', 'x', 'Easting'])
    h_y = resolve_col(df, ['Y', 'y', 'Northing'])
    h_z = resolve_col(df, ['Z', 'z', 'TVD'])
    h_gr = resolve_col(df, ['GR', 'gr', 'Gamma', 'GammaRay'])
    h_tvt_input = resolve_col(df, ['TVT_input', 'tvt_input', 'TVT'])
    
    t_tvt = resolve_col(t_df, ['TVT', 'tvt', 'MD', 'Depth'])
    t_gr = resolve_col(t_df, ['GR', 'gr', 'Gamma', 'GammaRay'])
    t_geo = resolve_col(t_df, ['Geology', 'geology', 'Formation', 'Zone'])
    
    # --- Robust NaN-Guarding Logic ---
    # Drop NaN rows from Reference log to prevent standard deviation propagation errors
    t_df = t_df.dropna(subset=[t_tvt, t_gr])
    
    # Interpolate NaN values in trajectory coordinates and observed log
    X_cols = [h_x, h_y, h_z]
    df[X_cols] = df[X_cols].interpolate(method='linear').bfill().ffill()
    df[h_gr] = df[h_gr].interpolate(method='linear').bfill().ffill()
    
    known_mask = df[h_tvt_input].notna()
    eval_mask = df[h_tvt_input].isna()
    eval_indices = np.where(eval_mask.values)[0]
    
    if len(eval_indices) == 0:
        continue
        
    known_df = df[known_mask]
    eval_df = df[eval_mask]
    
    X_train_raw = known_df[X_cols]
    y_train = known_df[h_tvt_input]
    X_eval_raw = eval_df[X_cols]
    
    # 1. Transform spatial polynomial features (degree 2)
    poly = PolynomialFeatures(degree=2, include_bias=False)
    X_train_poly = poly.fit_transform(X_train_raw)
    X_eval_poly = poly.transform(X_eval_raw)
    
    # 2. Scale features for Ridge regularization stability
    mean = X_train_poly.mean(axis=0)
    std = X_train_poly.std(axis=0)
    std[std == 0] = 1.0
    
    X_train_scaled = (X_train_poly - mean) / std
    X_eval_scaled = (X_eval_poly - mean) / std
    
    # 3. Fit Ridge Regression locally
    model = Ridge(alpha=10.0)
    model.fit(X_train_scaled, y_train)
    tvt_trend = model.predict(X_eval_scaled)
    
    # 4. Setup reference functions from vertical Typewell
    tw_tvt_vals = t_df[t_tvt].values.astype(np.float64)
    tw_gr_vals = t_df[t_gr].values.astype(np.float64)
    si = np.argsort(tw_tvt_vals)
    tw_tvt_vals, tw_gr_vals = tw_tvt_vals[si], tw_gr_vals[si]
    
    # Standardize both observed and typewell logs using the Typewell statistics
    tw_gr_mean, tw_gr_std = tw_gr_vals.mean(), max(tw_gr_vals.std(), 1.0)
    tw_gr_scaled = np.clip((tw_gr_vals - tw_gr_mean) / tw_gr_std, -3.0, 3.0)
    interp_gr_scaled = interp1d(tw_tvt_vals, tw_gr_scaled, kind='linear', bounds_error=False, fill_value='extrapolate')
    
    if t_geo and t_geo in t_df.columns:
        tw_geo_vals = t_df[t_geo].values[si]
        interp_geo = interp1d(tw_tvt_vals, np.arange(len(tw_geo_vals)), kind='nearest', bounds_error=False, fill_value='extrapolate')
    else:
        interp_geo = None
        
    obs_gr = eval_df[h_gr].values
    obs_gr_scaled = np.clip((obs_gr - tw_gr_mean) / tw_gr_std, -3.0, 3.0)
    
    # --- 5. Extended Kalman Filter (EKF) live tracking loop ---
    n_eval = len(eval_df)
    live_preds = np.zeros(n_eval)
    
    # Filter hyper-parameters
    Q = 0.015**2  # Process noise variance (drift rate)
    R = 0.85**2   # Measurement noise variance (log variance)
    
    # Initialize state (offset relative to trend plane)
    # At boundary start, offset is initialized from last known value
    last_known_tvt = df[h_tvt_input].values[eval_indices[0] - 1]
    last_poly = poly.transform(known_df[X_cols].iloc[[-1]])
    last_trend_val = model.predict((last_poly - mean) / std)[0]
    x_state = last_known_tvt - last_trend_val
    P_cov = 1.0
    
    for k in range(n_eval):
        # 1. State Prediction
        x_pred = x_state
        P_pred = P_cov + Q
        
        # Predicted TVT
        tvt_pred_step = tvt_trend[k] + x_pred
        
        # 2. Measurement Prediction & Local Gradient
        gr_pred = interp_gr_scaled(tvt_pred_step)
        
        # Local derivative of Gamma Ray curve at predicted TVT
        h_delta = 0.2
        gr_plus = interp_gr_scaled(tvt_pred_step + h_delta)
        gr_minus = interp_gr_scaled(tvt_pred_step - h_delta)
        H_grad = (gr_plus - gr_minus) / (2.0 * h_delta)
        
        # 3. Kalman Update
        innov = obs_gr_scaled[k] - gr_pred
        S_cov = H_grad * H_grad * P_pred + R
        K_gain = (P_pred * H_grad) / S_cov
        
        # State correction
        x_state = x_pred + K_gain * innov
        P_cov = (1.0 - K_gain * H_grad) * P_pred
        
        # Final prediction cache
        live_preds[k] = tvt_trend[k] + x_state
        
    # Apply mild localized smoothing for physical trajectory constraints
    if len(live_preds) > 15:
        live_preds = savgol_filter(live_preds, window_length=11, polyorder=2)
        
    # --- Live Geosteering Dashboards logs ---
    centroid = X_train_raw.mean(axis=0).values
    spatial_range = np.maximum(X_train_raw.std(axis=0).values, 1.0)
    distances = np.sqrt(np.sum(((X_eval_raw.values - centroid) / spatial_range) ** 2, axis=1))
    spatial_conf = np.exp(-0.02 * distances)
    
    ref_gr_at_preds = interp_gr_scaled(live_preds)
    gr_mismatch = np.abs(obs_gr_scaled - ref_gr_at_preds)
    gr_conf = np.exp(-0.5 * gr_mismatch)
    pcs = 100.0 * spatial_conf * gr_conf
    
    if interp_geo:
        geo_indices = interp_geo(live_preds).astype(int)
        predicted_geology = tw_geo_vals[np.clip(geo_indices, 0, len(tw_geo_vals)-1)]
        in_target = np.isin(predicted_geology, ['EGFDL', 'BUDA'])
        pct_in_target = 100.0 * np.sum(in_target) / len(live_preds)
    else:
        predicted_geology = np.full(len(live_preds), 'UNKNOWN')
        pct_in_target = 0.0
        
    print(f'\n[+] Live Geosteering Well {wellname}:')
    print(f'    - Real-Time TVT Range: [{live_preds.min():.2f}, {live_preds.max():.2f}]')
    print(f'    - Average PCS (Prediction Confidence): {pcs.mean():.1f}%')
    print(f'    - Time spent in target reservoir: {pct_in_target:.1f}%')
    
    for i, idx in enumerate(eval_indices):
        submission_rows.append({'id': f'{wellname}_{idx}', 'tvt': live_preds[i]})

# ============================================================
# SAVE SUBMISSION
# ============================================================
sub_df = pd.DataFrame(submission_rows)
sub_df.to_csv('submission.csv', index=False)
print(f'\n[+] Saved submission.csv containing {len(sub_df)} rows.')
print(f'    Predictions summary: mean={sub_df["tvt"].mean():.2f}, std={sub_df["tvt"].std():.2f}')
print('--- Inference completed successfully ---')